# 자녀 가치관 기반 유저 매칭 추천 시스템

이 노트북은 설문조사 데이터를 기반으로 선호동물상 외관과 자녀 가치관을 분석하여  
**유사도 기반 추천**과 **보완도 기반 추천**을 제공하는 매칭 시스템을 구현합니다.

---
## 목차
1. 데이터 불러오기
2. 데이터 전처리
3. 피처 엔지니어링 (One-hot, 중요도 가중치, MBTI형 성향 점수)
4. 유사도 기반 매칭 추천
5. 보완도 기반 매칭 추천
6. 통합 추천 시스템
---

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기

In [4]:
# CSV 파일 경로 설정 (필요시 수정)
FILE_PATH = "../data/저출산_소개팅_설문조사_더미데이터만_9966rows.csv"

df = pd.read_csv(FILE_PATH)
print(f"데이터 shape: {df.shape}")
df.head()

데이터 shape: (9966, 24)


,타임스탬프,0. 당신의 성함,0. 당신의 이상형,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.,...,맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도",Unnamed: 23
0,2026/02/24 5:37:18 AM GMT+9,윤서나,오리상,1명,"딸 1명, 아들 1명",경제적 안정 후,의학적 도움 적극 시도,2,5,4,...,4,4,5,1,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",4,NaN
1,2026/02/25 7:42:53 AM GMT+9,권은준,고양이상,1명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,입양 고려,4,3,4,...,4,4,3,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",2,NaN
2,2026/02/20 6:48:58 AM GMT+9,강린지,강아지상,2명,오직 딸,결혼 후 1~2년 이내,입양 고려;무자녀;의학적 도움 적극 시도,4,1,5,...,4,4,4,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",3,"회복탄력성, 생활력 강한 사람",2,NaN
3,2026/03/03 6:06:42 PM GMT+9,강태성,곰상,3명,"딸 2명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,2,5,...,5,2,5,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3,NaN
4,2026/02/27 2:51:11 PM GMT+9,이경온,여우상,1명,오직 딸,경제적 안정 후,무자녀,4,2,5,...,4,2,2,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",4,"회복탄력성, 생활력 강한 사람",5,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9966 entries, 0 to 9965
Data columns (total 24 columns):
 #   Column                                                                                 Non-Null Count  Dtype  
---  ------                                                                                 --------------  -----  
 0   타임스탬프                                                                                  9966 non-null   str    
 1   0. 당신의 성함                                                                              9966 non-null   str    
 2   0. 당신의 이상형                                                                             9966 non-null   str    
 3   1-1. 희망하는 자녀 수                                                                         9966 non-null   str    
 4   1-2. 희망하는 자녀 구성                                                                        9966 non-null   str    
 5   1-3. 자녀 갖고 싶은 시기                                                                       

In [6]:
# 원본 데이터 백업
df_raw = df.copy()

# 2. 데이터 전처리

## (1) 컬럼명 정리

In [7]:
def clean_colname(col: str) -> str:
    """컬럼명 정규화: 공백/따옴표/줄바꿈 정리"""
    col = str(col).strip()
    col = col.replace("\n", " ").replace("\t", " ")
    col = re.sub(r"\s+", " ", col)
    col = col.replace('"', "")
    return col

# 정규화 적용
cleaned_cols = [clean_colname(c) for c in df.columns]

# 중복 컬럼명 처리
seen = {}
unique_cols = []
for c in cleaned_cols:
    if c in seen:
        seen[c] += 1
        unique_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        unique_cols.append(c)

df.columns = unique_cols
print("정리된 컬럼 목록:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

정리된 컬럼 목록:
  0: 타임스탬프
  1: 0. 당신의 성함
  2: 0. 당신의 이상형
  3: 1-1. 희망하는 자녀 수
  4: 1-2. 희망하는 자녀 구성
  5: 1-3. 자녀 갖고 싶은 시기
  6: 1-4. 생물학적 출산이 어려움 발생 시 대안
  7: 1. 자녀 계획 및 가족 구성 항목에 대해 중요도
  8: 아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?
  9: 평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.
  10: 경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?
  11: 재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?
  12: 두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?
  13: 한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.
  14: 맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?
  15: 양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?
  16: 주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?
  17: 아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?
  18: 4-1. 자녀 교육비/양육비 지출 비중
  19: 4-2. 육아 휴직, 양육 부담
  20: 4. 경제적 지원 및 가사 분담에 대해 중요도
  21: 5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?
  22: 5. 자녀 가치관에 대해 중요도
  23: Unnamed: 23


## (2) 컬럼 리네임

In [8]:
# 컬럼 리네임 매핑
RENAME_MAP = {
    '0. 당신의 성함': 'user_name',
    '0. 당신의 이상형': 'ideal_type',
    '1-1. 희망하는 자녀 수': 'p_children_count',
    '1-2. 희망하는 자녀 구성': 'p_children_composition',
    '1-3. 자녀 갖고 싶은 시기': 'p_children_timing',
    '1-4. 생물학적 출산이 어려움 발생 시 대안': 'p_infertility_alternative',
    '1. 자녀 계획 및 가족 구성 항목에 대해 중요도': 'imp_family_plan',
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?': 'sc_toothbrushing',
    '평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.': 'sc_bedtime_story',
    '경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?': 'sc_competition_2nd',
    '재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?': 'sc_talent_education',
    '두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?': 'sc_discipline_conflict',
    '한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.': 'sc_play_vs_chores',
    '맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?': 'sc_grandparents_help',
    '양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?': 'sc_inlaws_advice',
    '주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?': 'sc_rainy_zoo',
    '아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?': 'sc_education_fund_risk',
    '4-1. 자녀 교육비/양육비 지출 비중': 'e_childcare_cost_share',
    '4-2. 육아 휴직, 양육 부담': 'e_parental_leave_burden',
    '4. 경제적 지원 및 가사 분담에 대해 중요도': 'imp_econ_housework',
    '5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?': 'child_values_open',
    '5. 자녀 가치관에 대해 중요도': 'imp_child_values'
}

# 정확한 매칭이 안 되는 경우를 위한 부분 매칭
def find_and_rename(df, rename_map):
    new_rename = {}
    for old_name, new_name in rename_map.items():
        # 정확한 매칭 시도
        if old_name in df.columns:
            new_rename[old_name] = new_name
        else:
            # 부분 매칭 시도 (핵심 키워드로)
            for col in df.columns:
                # 핵심 키워드 추출하여 매칭
                key_parts = old_name.split()[:3]  # 앞 3단어
                if all(part in col for part in key_parts[:2]):
                    new_rename[col] = new_name
                    break
    return new_rename

actual_rename = find_and_rename(df, RENAME_MAP)
df = df.rename(columns=actual_rename)

print(f"리네임된 컬럼 수: {len(actual_rename)}")
print("\n현재 컬럼 목록:")
print(list(df.columns))

리네임된 컬럼 수: 22

현재 컬럼 목록:
['타임스탬프', 'user_name', 'ideal_type', 'p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative', 'imp_family_plan', 'sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk', 'e_childcare_cost_share', 'e_parental_leave_burden', 'imp_econ_housework', 'child_values_open', 'imp_child_values', 'Unnamed: 23']


## (3) 불필요한 컬럼 제거 및 인덱스 설정

In [9]:
# 타임스탬프 및 불필요한 컬럼 제거
drop_cols = ['타임스탬프']
# Unnamed 컬럼도 제거
drop_cols += [c for c in df.columns if 'Unnamed' in c]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# user_name을 인덱스로 설정하기 전에 중복 확인
if 'user_name' in df.columns:
    # 중복된 user_name이 있으면 번호 붙이기
    if df['user_name'].duplicated().any():
        print("⚠️ 중복된 user_name 발견, 고유하게 변환합니다.")
        name_counts = {}
        unique_names = []
        for name in df['user_name']:
            if name in name_counts:
                name_counts[name] += 1
                unique_names.append(f"{name}_{name_counts[name]}")
            else:
                name_counts[name] = 0
                unique_names.append(name)
        df['user_name'] = unique_names
    
    df = df.set_index('user_name')

print(f"최종 데이터 shape: {df.shape}")
print(f"인덱스 고유값 수: {df.index.nunique()} / 전체: {len(df.index)}")
df.head()

⚠️ 중복된 user_name 발견, 고유하게 변환합니다.
최종 데이터 shape: (9966, 21)
인덱스 고유값 수: 9966 / 전체: 9966


,ideal_type,p_children_count,p_children_composition,p_children_timing,p_infertility_alternative,imp_family_plan,sc_toothbrushing,sc_bedtime_story,sc_competition_2nd,sc_talent_education,...,sc_play_vs_chores,sc_grandparents_help,sc_inlaws_advice,sc_rainy_zoo,sc_education_fund_risk,e_childcare_cost_share,e_parental_leave_burden,imp_econ_housework,child_values_open,imp_child_values
user_name,,,,,,,,,,,,,,,,,,,,,
윤서나,오리상,1명,"딸 1명, 아들 1명",경제적 안정 후,의학적 도움 적극 시도,2,5,4,4,2,...,2,4,4,5,1,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",4
권은준,고양이상,1명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,입양 고려,4,3,4,4,2,...,4,4,4,3,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",2
강린지,강아지상,2명,오직 딸,결혼 후 1~2년 이내,입양 고려;무자녀;의학적 도움 적극 시도,4,1,5,5,4,...,2,4,4,4,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",3,"회복탄력성, 생활력 강한 사람",2
강태성,곰상,3명,"딸 2명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,2,5,5,5,...,5,5,2,5,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3
이경온,여우상,1명,오직 딸,경제적 안정 후,무자녀,4,2,5,4,3,...,2,4,2,2,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",4,"회복탄력성, 생활력 강한 사람",5


## (4) 컬럼 그룹 정의

In [10]:
# 컬럼 그룹 정의
META_COLS = ['ideal_type']

PLAN_COLS = ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']

SCENARIO_COLS = ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education',
                 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 
                 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']

ECON_COLS = ['e_childcare_cost_share', 'e_parental_leave_burden']

TEXT_COLS = ['child_values_open']

IMPORTANCE_COLS = ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']

# 존재하는 컬럼만 필터링
all_groups = {
    'META_COLS': [c for c in META_COLS if c in df.columns],
    'PLAN_COLS': [c for c in PLAN_COLS if c in df.columns],
    'SCENARIO_COLS': [c for c in SCENARIO_COLS if c in df.columns],
    'ECON_COLS': [c for c in ECON_COLS if c in df.columns],
    'TEXT_COLS': [c for c in TEXT_COLS if c in df.columns],
    'IMPORTANCE_COLS': [c for c in IMPORTANCE_COLS if c in df.columns]
}

for gname, cols in all_groups.items():
    print(f"{gname}: {len(cols)}개 - {cols}")

META_COLS: 1개 - ['ideal_type']
PLAN_COLS: 4개 - ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']
SCENARIO_COLS: 10개 - ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']
ECON_COLS: 2개 - ['e_childcare_cost_share', 'e_parental_leave_burden']
TEXT_COLS: 1개 - ['child_values_open']
IMPORTANCE_COLS: 3개 - ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']


# 3. 피처 엔지니어링

## (1) 범주형 변수 One-Hot Encoding (중요도 가중치 적용)

In [11]:
def one_hot_with_importance(df, col, importance_col=None, known_categories=None):
    """
    범주형 컬럼을 One-Hot 인코딩하고, 중요도 가중치를 적용합니다.
    복수 응답(세미콜론 구분)도 처리합니다.
    
    ★ 수정: iloc 기반 인덱싱으로 중복 인덱스 문제 해결
    """
    if col not in df.columns:
        return pd.DataFrame(index=df.index)
    
    # 복수 응답 분리 및 카테고리 수집
    all_categories = set()
    for val in df[col].dropna():
        for item in str(val).split(';'):
            item = item.strip()
            if item:
                all_categories.add(item)
    
    # 알려진 카테고리가 있으면 그 외는 '기타'로
    if known_categories:
        final_categories = list(known_categories) + ['기타']
    else:
        final_categories = list(all_categories)
    
    # One-Hot 인코딩 결과 초기화
    result = pd.DataFrame(0.0, index=df.index, columns=[f"{col}_{cat}" for cat in final_categories])
    
    # 중요도 가중치 (1~5점을 0.2~1.0으로 변환)
    if importance_col and importance_col in df.columns:
        weights = df[importance_col].fillna(3).astype(float) / 5.0
    else:
        weights = pd.Series(1.0, index=df.index)
    
    # ★ 수정: iloc 기반으로 행 순회 (중복 인덱스 문제 해결)
    for i in range(len(df)):
        idx = df.index[i]  # 실제 인덱스 값
        val = df.iloc[i][col]  # iloc으로 값 가져오기
        
        # 스칼라 값인지 확인 후 결측 체크
        if pd.isna(val):
            continue
        
        val_str = str(val)
        items = [item.strip() for item in val_str.split(';') if item.strip()]
        n_items = len(items)
        
        if n_items == 0:
            continue
        
        # 가중치도 iloc으로 가져오기
        weight = float(weights.iloc[i])
        
        for item in items:
            if known_categories and item not in known_categories:
                col_name = f"{col}_기타"
            else:
                col_name = f"{col}_{item}"
            
            if col_name in result.columns:
                # ★ iloc으로 값 설정
                result.iloc[i, result.columns.get_loc(col_name)] = weight / n_items
    
    return result

In [12]:
# 알려진 카테고리 정의
KNOWN_CATEGORIES = {
    'p_children_count': ['1명', '2명', '3명', '그 이상'],
    'p_children_composition': ['오직 딸', '오직 아들', '딸 1명, 아들 1명'],
    'p_children_timing': ['결혼 즉시', '결혼 후 1~2년 이내', '결혼 후 3~5년 이내', '경제적 안정 후'],
    'p_infertility_alternative': ['의학적 도움 적극 시도', '입양 고려', '무자녀'],
    'e_childcare_cost_share': ['노후보단 자녀 교육', '노후 먼저, 남는 예산으로 지원'],
    'e_parental_leave_burden': ['경제력 높은 사람 일하고, 한명은 전담 육아', '맞벌이하면서 외부 도움(조부모, 시터)'],
    'child_values_open': ['경제적 성공, 사회적 지위', '도덕적, 타인 배려', '자신이 좋아하는 일, 행복', '회복탄력성, 생활력 강한 사람']
}

# 중요도 매핑
IMPORTANCE_MAPPING = {
    'p_children_count': 'imp_family_plan',
    'p_children_composition': 'imp_family_plan',
    'p_children_timing': 'imp_family_plan',
    'p_infertility_alternative': 'imp_family_plan',
    'e_childcare_cost_share': 'imp_econ_housework',
    'e_parental_leave_burden': 'imp_econ_housework',
    'child_values_open': 'imp_child_values'
}

# One-Hot Encoding 수행
onehot_dfs = []
for col, categories in KNOWN_CATEGORIES.items():
    if col in df.columns:
        imp_col = IMPORTANCE_MAPPING.get(col)
        oh_df = one_hot_with_importance(df, col, imp_col, categories)
        onehot_dfs.append(oh_df)
        print(f"{col}: {oh_df.shape[1]}개 피처 생성")

# 모든 One-Hot 컬럼 합치기
if onehot_dfs:
    df_onehot = pd.concat(onehot_dfs, axis=1)
else:
    df_onehot = pd.DataFrame(index=df.index)
    
print(f"\n총 One-Hot 피처 수: {df_onehot.shape[1]}")
df_onehot.head()

p_children_count: 5개 피처 생성
p_children_composition: 4개 피처 생성
p_children_timing: 5개 피처 생성
p_infertility_alternative: 4개 피처 생성
e_childcare_cost_share: 3개 피처 생성
e_parental_leave_burden: 3개 피처 생성
child_values_open: 5개 피처 생성

총 One-Hot 피처 수: 29


,p_children_count_1명,p_children_count_2명,p_children_count_3명,p_children_count_그 이상,p_children_count_기타,p_children_composition_오직 딸,p_children_composition_오직 아들,"p_children_composition_딸 1명, 아들 1명",p_children_composition_기타,p_children_timing_결혼 즉시,...,"e_childcare_cost_share_노후 먼저, 남는 예산으로 지원",e_childcare_cost_share_기타,"e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아","e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)",e_parental_leave_burden_기타,"child_values_open_경제적 성공, 사회적 지위","child_values_open_도덕적, 타인 배려","child_values_open_자신이 좋아하는 일, 행복","child_values_open_회복탄력성, 생활력 강한 사람",child_values_open_기타
user_name,,,,,,,,,,,,,,,,,,,,,
윤서나,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,...,0.4,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.8,0.0
권은준,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,...,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.4,0.0
강린지,0.0,0.8,0.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0,...,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.4,0.0
강태성,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.2,0.0,...,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.6,0.0,0.0
이경온,0.8,0.0,0.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0,1.0,0.0


## (2) 이상형 동물상 One-Hot Encoding

In [13]:
# 이상형 동물상 One-Hot (가중치 없음, 단순 매칭용)
KNOWN_IDEAL_TYPES = ['강아지상', '고양이상', '사슴상', '토끼상', '꼬부기상', '곰상', '쿼카상', 
                      '쥐상', '도룡뇽상', '여우상', '너구리상', '오리상', '꽃돼지상']

if 'ideal_type' in df.columns:
    df_ideal = pd.get_dummies(df['ideal_type'], prefix='ideal_type')
    # 알려지지 않은 동물상 처리
    for animal in KNOWN_IDEAL_TYPES:
        col_name = f'ideal_type_{animal}'
        if col_name not in df_ideal.columns:
            df_ideal[col_name] = 0
    print(f"이상형 동물상 피처 수: {df_ideal.shape[1]}")
else:
    df_ideal = pd.DataFrame(index=df.index)

df_ideal.head()

이상형 동물상 피처 수: 22


,ideal_type_강아지상,ideal_type_고슴도치상,ideal_type_고양이상,ideal_type_곰상,ideal_type_공룡상,ideal_type_꼬부기상,ideal_type_꽃돼지상,ideal_type_너구리상,ideal_type_늑대상,ideal_type_다람쥐상,...,ideal_type_여우상,ideal_type_오리상,ideal_type_쿼카상,ideal_type_토끼상,ideal_type_판다상,ideal_type_펭귄상,ideal_type_햄스터상,ideal_type_호랑이상,ideal_type_쥐상,ideal_type_도룡뇽상
user_name,,,,,,,,,,,,,,,,,,,,,
윤서나,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,0,0
권은준,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,0,0
강린지,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,0,0
강태성,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,0,0
이경온,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,0,0


## (3) 시나리오 응답 기반 MBTI형 성향 점수 계산

5개 축:
- **S-F** (Strict-Flexible): 양육 실행 스타일
- **A-H** (Achievement-Happiness): 교육/성장 우선순위  
- **E-L** (Equal-Lead): 공동양육 운영 방식
- **B-T** (Boundary-Together): 확장가족/경계
- **P-G** (Plan-Go): 계획/리스크 대응

In [14]:
# 시나리오 컬럼이 존재하는지 확인
scenario_cols = [c for c in SCENARIO_COLS if c in df.columns]
print(f"찾은 시나리오 컬럼 수: {len(scenario_cols)}")
print(scenario_cols)

찾은 시나리오 컬럼 수: 10
['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']


In [15]:
# 시나리오 문항을 2개씩 묶어서 5개 축으로
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

# 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름)
AXES = [
    ("S", "F", "parenting_style_SF"),      # 양육 실행 스타일
    ("A", "H", "education_priority_AH"),   # 교육/성장 우선순위
    ("E", "L", "co_parenting_mode_EL"),    # 공동양육 운영 방식
    ("B", "T", "family_boundary_BT"),      # 확장가족/경계
    ("P", "G", "planning_risk_PG"),        # 계획/리스크 대응
]

print(f"축 개수: {len(AXES)}, 페어 개수: {len(pairs)}")
for i, (pair, axis) in enumerate(zip(pairs, AXES)):
    print(f"  축 {i+1} ({axis[2]}): {pair[0]} + {pair[1]}")

축 개수: 5, 페어 개수: 5
  축 1 (parenting_style_SF): sc_toothbrushing + sc_bedtime_story
  축 2 (education_priority_AH): sc_competition_2nd + sc_talent_education
  축 3 (co_parenting_mode_EL): sc_discipline_conflict + sc_play_vs_chores
  축 4 (family_boundary_BT): sc_grandparents_help + sc_inlaws_advice
  축 5 (planning_risk_PG): sc_rainy_zoo + sc_education_fund_risk


In [16]:
def calculate_trait_scores(df, pairs, axes):
    """
    시나리오 응답으로부터 성향 점수 계산
    응답: 1-5점 (1=왼쪽 성향, 5=오른쪽 성향, 3=중립)
    """
    trait_scores = {}
    mbti_letters = {}
    
    for i, ((col1, col2), (left, right, axis_name)) in enumerate(zip(pairs, axes)):
        if col1 not in df.columns or col2 not in df.columns:
            continue
        
        # 응답값 합산 (결측값은 3으로 대체)
        v1 = pd.to_numeric(df[col1], errors='coerce').fillna(3).astype(float)
        v2 = pd.to_numeric(df[col2], errors='coerce').fillna(3).astype(float)
        
        # 두 문항 합산 (2~10점 범위)
        total = v1 + v2
        
        # 0~1 범위로 정규화
        trait_scores[axis_name] = (total - 2) / 8.0
        
        # MBTI 스타일 문자 결정 (각 값을 개별적으로 판단)
        def get_letter(x):
            if x < 6:
                return left
            elif x > 6:
                return right
            else:
                return left  # 중립은 왼쪽으로
        
        mbti_letters[axis_name + '_letter'] = total.apply(get_letter)
    
    return pd.DataFrame(trait_scores), pd.DataFrame(mbti_letters)

df_traits, df_mbti = calculate_trait_scores(df, pairs, AXES)
print(f"성향 점수 shape: {df_traits.shape}")
df_traits.head()

성향 점수 shape: (9966, 5)


,parenting_style_SF,education_priority_AH,co_parenting_mode_EL,family_boundary_BT,planning_risk_PG
user_name,,,,,
윤서나,0.875,0.500,0.125,0.750,0.50
권은준,0.625,0.500,0.750,0.750,0.50
강린지,0.500,0.875,0.125,0.750,0.50
강태성,0.625,1.000,1.000,0.625,0.75
이경온,0.625,0.625,0.500,0.500,0.25


In [17]:
# MBTI 유형 조합 생성
def create_mbti_type(row):
    letters = []
    for col in row.index:
        val = row[col]
        if pd.notna(val):
            letters.append(str(val))
    return ''.join(letters)

df_mbti['childcare_mbti'] = df_mbti.apply(create_mbti_type, axis=1)
print("자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts())

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    1006
SHLBP     773
SHEBG     749
SHEBP     712
SHLTG     554
FHLTG     501
FHEBP     478
SHETG     477
FHLBG     462
FHLBP     460
FHEBG     418
SHLTP     370
SHETP     360
SALBP     343
FHETG     298
FHLTP     293
SALTP     259
SALBG     217
SAEBG     167
FHETP     158
SAEBP     139
SAETP     101
FALTG     100
FALBP      88
SALTG      81
SAETG      75
FAEBP      70
FAEBG      66
FALTP      64
FALBG      55
FAETG      52
FAETP      20
Name: count, dtype: int64


## (4) 모든 피처 합치기

In [18]:
# 모든 피처 합치기
df_features = pd.concat([df_onehot, df_traits], axis=1)

# 결측값 처리
df_features = df_features.fillna(0)

print(f"최종 피처 매트릭스 shape: {df_features.shape}")
print(f"\n피처 목록:")
for i, col in enumerate(df_features.columns):
    print(f"  {i+1}. {col}")

최종 피처 매트릭스 shape: (9966, 34)

피처 목록:
  1. p_children_count_1명
  2. p_children_count_2명
  3. p_children_count_3명
  4. p_children_count_그 이상
  5. p_children_count_기타
  6. p_children_composition_오직 딸
  7. p_children_composition_오직 아들
  8. p_children_composition_딸 1명, 아들 1명
  9. p_children_composition_기타
  10. p_children_timing_결혼 즉시
  11. p_children_timing_결혼 후 1~2년 이내
  12. p_children_timing_결혼 후 3~5년 이내
  13. p_children_timing_경제적 안정 후
  14. p_children_timing_기타
  15. p_infertility_alternative_의학적 도움 적극 시도
  16. p_infertility_alternative_입양 고려
  17. p_infertility_alternative_무자녀
  18. p_infertility_alternative_기타
  19. e_childcare_cost_share_노후보단 자녀 교육
  20. e_childcare_cost_share_노후 먼저, 남는 예산으로 지원
  21. e_childcare_cost_share_기타
  22. e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아
  23. e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)
  24. e_parental_leave_burden_기타
  25. child_values_open_경제적 성공, 사회적 지위
  26. child_values_open_도덕적, 타인 배려
  27. child_values_open_자신이 좋아하는 일, 행복
  28. ch

In [19]:
# 피처 그룹 정의 (추천 시스템에서 사용)
FEATURE_GROUPS = {
    'value_cols': [c for c in df_features.columns if c.startswith(('p_', 'e_', 'child_'))],
    'trait_cols': [c for c in df_features.columns if c in ['parenting_style_SF', 'education_priority_AH', 
                                                           'co_parenting_mode_EL', 'family_boundary_BT', 
                                                           'planning_risk_PG']]
}

print(f"가치관 피처 수: {len(FEATURE_GROUPS['value_cols'])}")
print(f"성향 피처 수: {len(FEATURE_GROUPS['trait_cols'])}")

가치관 피처 수: 29
성향 피처 수: 5


# 4. 유사도 기반 매칭 추천

코사인 유사도를 사용하여 가치관이 비슷한 사람을 추천합니다.

In [20]:
# 가치관 유사도 매트릭스 계산
value_cols = FEATURE_GROUPS['value_cols']
if len(value_cols) > 0:
    val_sim_matrix = cosine_similarity(df_features[value_cols])
    val_sim_df = pd.DataFrame(val_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"가치관 유사도 매트릭스 shape: {val_sim_df.shape}")
else:
    val_sim_df = pd.DataFrame(1.0, index=df_features.index, columns=df_features.index)

val_sim_df.head()

가치관 유사도 매트릭스 shape: (9966, 9966)


user_name,윤서나,권은준,강린지,강태성,이경온,권훈나,김연린,송하태,조수도,문아주,...,서림경_1,전준린,남주지,백담라,유훈나,정수현,허호하,허림수,남별훈,유태호_1
user_name,,,,,,,,,,,,,,,,,,,,,
윤서나,1.000000,0.507833,0.303616,0.163299,0.517464,0.298347,0.316228,0.622841,0.239046,0.645772,...,0.282843,0.478091,0.124568,0.217643,0.197814,0.180702,0.343559,0.219265,0.388449,0.319505
권은준,0.507833,1.000000,0.493397,0.236940,0.354552,0.072148,0.573539,0.421733,0.086711,0.435028,...,0.000000,0.346844,0.632599,0.210526,0.191346,0.327737,0.041541,0.063628,0.086711,0.430473
강린지,0.303616,0.493397,1.000000,0.337146,0.453875,0.523369,0.608076,0.645478,0.537079,0.448153,...,0.480904,0.493532,0.378209,0.281941,0.544540,0.405993,0.528505,0.149122,0.399180,0.194014
강태성,0.163299,0.236940,0.337146,1.000000,0.093891,0.000000,0.215166,0.305129,0.487950,0.075324,...,0.057735,0.585540,0.305129,0.177705,0.430706,0.221313,0.116881,0.429669,0.170783,0.596285
이경온,0.517464,0.354552,0.453875,0.093891,1.000000,0.562262,0.818182,0.501352,0.549767,0.914971,...,0.487869,0.274883,0.429730,0.250272,0.379117,0.415584,0.345682,0.302564,0.429505,0.524864


In [21]:
# 성향 유사도 매트릭스 계산
trait_cols = FEATURE_GROUPS['trait_cols']
if len(trait_cols) > 0:
    trait_sim_matrix = cosine_similarity(df_features[trait_cols])
    trait_sim_df = pd.DataFrame(trait_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"성향 유사도 매트릭스 shape: {trait_sim_df.shape}")
else:
    trait_sim_df = pd.DataFrame(1.0, index=df_features.index, columns=df_features.index)

trait_sim_df.head()

성향 유사도 매트릭스 shape: (9966, 9966)


user_name,윤서나,권은준,강린지,강태성,이경온,권훈나,김연린,송하태,조수도,문아주,...,서림경_1,전준린,남주지,백담라,유훈나,정수현,허호하,허림수,남별훈,유태호_1
user_name,,,,,,,,,,,,,,,,,,,,,
윤서나,1.000000,0.883468,0.923729,0.811786,0.903340,0.755427,0.855298,0.841668,0.975506,0.902445,...,0.722158,0.881638,0.683640,0.869235,0.754535,0.833449,0.744092,0.937382,0.720897,0.853207
권은준,0.883468,1.000000,0.859152,0.956964,0.958909,0.960599,0.961384,0.955918,0.940895,0.988402,...,0.794285,0.927533,0.917197,0.905012,0.943694,0.972490,0.957352,0.968660,0.890295,0.954216
강린지,0.923729,0.859152,1.000000,0.868423,0.903340,0.806934,0.908203,0.762762,0.925904,0.902445,...,0.776320,0.969802,0.712125,0.902244,0.819831,0.883456,0.797241,0.937382,0.783888,0.853207
강태성,0.811786,0.956964,0.868423,1.000000,0.950897,0.988043,0.995230,0.937489,0.908540,0.972760,...,0.911623,0.960187,0.965939,0.964109,0.991284,0.984034,0.980093,0.960092,0.971878,0.975888
이경온,0.903340,0.958909,0.903340,0.950897,1.000000,0.925102,0.970880,0.924281,0.968367,0.988892,...,0.824762,0.946659,0.878646,0.940859,0.943324,0.976272,0.933859,0.959186,0.926416,0.983296


In [22]:
def get_similarity_recommendations(user_name, top_n=5, value_weight=0.7, trait_weight=0.3):
    """
    유사도 기반 추천: 가치관과 성향이 비슷한 사람 추천
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    recommendations = []
    
    # 현재 사용자의 위치 (iloc용)
    user_iloc = df_features.index.get_loc(user_name)
    
    for i, other_user in enumerate(df_features.index):
        if user_name == other_user:
            continue
        
        # iloc으로 유사도 점수 가져오기 (스칼라 보장)
        val_score = float(val_sim_df.iloc[user_iloc, i])
        trait_score = float(trait_sim_df.iloc[user_iloc, i])
        
        # 종합 점수
        total_score = value_weight * val_score + trait_weight * trait_score
        
        # 이상형 매칭 여부 확인 (iloc 사용)
        ideal_match = False
        if 'ideal_type' in df.columns:
            my_ideal = df.iloc[user_iloc]['ideal_type'] if user_iloc < len(df) else None
            other_ideal = df.iloc[i]['ideal_type'] if i < len(df) else None
            if pd.notna(my_ideal) and pd.notna(other_ideal):
                ideal_match = bool(str(my_ideal) == str(other_ideal))
        
        # MBTI 유형 가져오기 (iloc 사용)
        other_mbti = 'N/A'
        if 'childcare_mbti' in df_mbti.columns and i < len(df_mbti):
            mbti_val = df_mbti.iloc[i]['childcare_mbti']
            other_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'
        
        recommendations.append({
            'name': other_user,
            'value_similarity': round(val_score, 4),
            'trait_similarity': round(trait_score, 4),
            'total_score': round(total_score, 4),
            'childcare_mbti': other_mbti,
            'ideal_type_match': ideal_match,
            'match_type': '유사형(Similar)'
        })
    
    result_df = pd.DataFrame(recommendations)
    result_df = result_df.sort_values('total_score', ascending=False).head(top_n)
    result_df = result_df.reset_index(drop=True)
    result_df.index = result_df.index + 1  # 1부터 시작
    
    return result_df

# 테스트
test_user = df_features.index[0]
print(f"\n=== {test_user}님의 유사도 기반 추천 ===")
get_similarity_recommendations(test_user, top_n=5)


=== 윤서나님의 유사도 기반 추천 ===


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,남지현,0.9634,0.8391,0.9261,SHLTG,False,유사형(Similar)
2,오은우,0.9035,0.8343,0.8828,SHLTP,False,유사형(Similar)
3,한지솔,0.9035,0.8300,0.8814,SHLTP,False,유사형(Similar)
4,이예성_1,0.8771,0.8871,0.8801,FHEBP,False,유사형(Similar)
5,백찬수,0.8733,0.8814,0.8757,FHLBP,False,유사형(Similar)


# 5. 보완도 기반 매칭 추천

**핵심 가치관은 유사하면서, 양육 스타일은 서로 보완하는 사람을 추천합니다.**

- 가치관 (자녀 수, 교육비, 자녀 가치관 등) → **유사할수록 좋음**
- 양육 성향 (SF, AH, EL, BT, PG 축) → **서로 다르면 보완적**

In [23]:
def calculate_complementary_score(user1_iloc, user2_iloc, df_features, trait_cols):
    """
    보완도 점수 계산 (iloc 기반)
    성향 점수의 차이가 적절할수록 보완도가 높음
    """
    if len(trait_cols) == 0:
        return 0.0, {}
    
    trait_details = {}
    complementary_scores = []
    
    for col in trait_cols:
        col_idx = df_features.columns.get_loc(col)
        v1 = float(df_features.iloc[user1_iloc, col_idx])
        v2 = float(df_features.iloc[user2_iloc, col_idx])
        diff = abs(v1 - v2)
        
        # 보완도 점수: 차이가 0.3~0.5일 때 가장 높음
        optimal_diff = 0.4
        comp_score = float(np.exp(-((diff - optimal_diff) ** 2) / 0.1))
        
        complementary_scores.append(comp_score)
        trait_details[col] = {
            'user1': round(v1, 3),
            'user2': round(v2, 3),
            'diff': round(diff, 3),
            'comp_score': round(comp_score, 3)
        }
    
    avg_complementary = float(np.mean(complementary_scores)) if complementary_scores else 0.0
    
    return avg_complementary, trait_details

In [24]:
def get_complementary_recommendations(user_name, top_n=5, value_threshold=0.5):
    """
    보완도 기반 추천: 가치관은 유사하면서 성향은 보완적인 사람 추천
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    trait_cols = FEATURE_GROUPS['trait_cols']
    recommendations = []
    
    # 현재 사용자의 위치
    user_iloc = df_features.index.get_loc(user_name)
    
    # 내 MBTI 가져오기
    my_mbti = 'N/A'
    if 'childcare_mbti' in df_mbti.columns and user_iloc < len(df_mbti):
        mbti_val = df_mbti.iloc[user_iloc]['childcare_mbti']
        my_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'
    
    for i, other_user in enumerate(df_features.index):
        if user_name == other_user:
            continue
        
        # 가치관 유사도 (iloc 사용)
        val_score = float(val_sim_df.iloc[user_iloc, i])
        
        # 가치관 유사도가 기준 이하면 스킵
        if val_score < value_threshold:
            continue
        
        # 보완도 계산
        comp_score, trait_details = calculate_complementary_score(
            user_iloc, i, df_features, trait_cols
        )
        
        # 종합 점수: 가치관 유사도 * 보완도
        total_score = val_score * comp_score
        
        # MBTI 유형 가져오기
        other_mbti = 'N/A'
        if 'childcare_mbti' in df_mbti.columns and i < len(df_mbti):
            mbti_val = df_mbti.iloc[i]['childcare_mbti']
            other_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'
        
        # 이상형 매칭 여부
        ideal_match = False
        if 'ideal_type' in df.columns:
            my_ideal = df.iloc[user_iloc]['ideal_type'] if user_iloc < len(df) else None
            other_ideal = df.iloc[i]['ideal_type'] if i < len(df) else None
            if pd.notna(my_ideal) and pd.notna(other_ideal):
                ideal_match = bool(str(my_ideal) == str(other_ideal))
        
        recommendations.append({
            'name': other_user,
            'value_similarity': round(val_score, 4),
            'complementary_score': round(comp_score, 4),
            'total_score': round(total_score, 4),
            'my_mbti': my_mbti,
            'their_mbti': other_mbti,
            'ideal_type_match': ideal_match,
            'match_type': '보완형(Complementary)'
        })
    
    if not recommendations:
        return f"가치관 유사도 {value_threshold} 이상인 매칭 대상이 없습니다."
    
    result_df = pd.DataFrame(recommendations)
    result_df = result_df.sort_values('total_score', ascending=False).head(top_n)
    result_df = result_df.reset_index(drop=True)
    result_df.index = result_df.index + 1
    
    return result_df

# 테스트
print(f"\n=== {test_user}님의 보완도 기반 추천 ===")
get_complementary_recommendations(test_user, top_n=5, value_threshold=0.3)


=== 윤서나님의 보완도 기반 추천 ===


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,고수라,0.8377,0.9369,0.7849,FAETP,SHEBP,False,보완형(Complementary)
2,허담혜,0.9057,0.8355,0.7568,FAETP,SHLTP,False,보완형(Complementary)
3,전아수,0.8377,0.8979,0.7522,FAETP,SHLBP,False,보완형(Complementary)
4,허도하,0.7842,0.9547,0.7486,FAETP,SHEBP,False,보완형(Complementary)
5,남경담,0.8721,0.8321,0.7256,FAETP,SHETG,False,보완형(Complementary)


# 6. 통합 추천 시스템

유사도 기반과 보완도 기반 추천을 모두 제공하는 통합 함수

In [25]:
def get_smart_recommendations(user_name, top_n=5, mode='both'):
    """
    스마트 추천 시스템: 유사도 + 보완도 기반 통합 추천
    
    Parameters:
    -----------
    user_name : str - 추천 대상 사용자
    top_n : int - 각 유형별 추천할 인원 수
    mode : str - 'similar', 'complementary', 'both'
    
    Returns:
    --------
    dict with recommendation DataFrames
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    results = {}
    user_iloc = df_features.index.get_loc(user_name)
    
    # 사용자 정보 출력
    print("="*60)
    print(f"🎯 {user_name}님을 위한 맞춤 추천")
    print("="*60)
    
    # 사용자 프로필
    if 'childcare_mbti' in df_mbti.columns and user_iloc < len(df_mbti):
        user_mbti = df_mbti.iloc[user_iloc]['childcare_mbti']
        print(f"\n📋 당신의 자녀양육 MBTI: {user_mbti}")
    
    if 'ideal_type' in df.columns and user_iloc < len(df):
        user_ideal = df.iloc[user_iloc]['ideal_type']
        print(f"💕 당신의 이상형: {user_ideal}")
    
    # 유사도 기반 추천
    if mode in ['similar', 'both']:
        print(f"\n\n🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람")
        print("-"*50)
        similar_recs = get_similarity_recommendations(user_name, top_n)
        results['similar'] = similar_recs
        if isinstance(similar_recs, pd.DataFrame):
            display(similar_recs)
        else:
            print(similar_recs)
    
    # 보완도 기반 추천
    if mode in ['complementary', 'both']:
        print(f"\n\n🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람")
        print("-"*50)
        comp_recs = get_complementary_recommendations(user_name, top_n, value_threshold=0.3)
        results['complementary'] = comp_recs
        if isinstance(comp_recs, pd.DataFrame):
            display(comp_recs)
        else:
            print(comp_recs)
    
    print("\n" + "="*60)
    
    return results

In [26]:
# 통합 추천 테스트
test_user = df_features.index[0]
results = get_smart_recommendations(test_user, top_n=5, mode='both')

🎯 윤서나님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FAETP
💕 당신의 이상형: 오리상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,남지현,0.9634,0.8391,0.9261,SHLTG,False,유사형(Similar)
2,오은우,0.9035,0.8343,0.8828,SHLTP,False,유사형(Similar)
3,한지솔,0.9035,0.8300,0.8814,SHLTP,False,유사형(Similar)
4,이예성_1,0.8771,0.8871,0.8801,FHEBP,False,유사형(Similar)
5,백찬수,0.8733,0.8814,0.8757,FHLBP,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,고수라,0.8377,0.9369,0.7849,FAETP,SHEBP,False,보완형(Complementary)
2,허담혜,0.9057,0.8355,0.7568,FAETP,SHLTP,False,보완형(Complementary)
3,전아수,0.8377,0.8979,0.7522,FAETP,SHLBP,False,보완형(Complementary)
4,허도하,0.7842,0.9547,0.7486,FAETP,SHEBP,False,보완형(Complementary)
5,남경담,0.8721,0.8321,0.7256,FAETP,SHETG,False,보완형(Complementary)


In [27]:
# 여러 사용자 테스트
print("\n" + "#"*70)
print("# 다른 사용자 추천 결과")
print("#"*70)

for user in list(df_features.index)[1:4]:  # 2~4번째 사용자
    print("\n")
    get_smart_recommendations(user, top_n=3, mode='both')


######################################################################
# 다른 사용자 추천 결과
######################################################################


🎯 권은준님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FALTP
💕 당신의 이상형: 고양이상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,강혜호,0.9398,0.9690,0.9485,FALBG,False,유사형(Similar)
2,강영서,0.9234,0.9743,0.9387,FHLBG,False,유사형(Similar)
3,배윤은,0.9139,0.9892,0.9365,FALTG,True,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,남경혜,0.8658,0.8143,0.7050,FALTP,SHETG,False,보완형(Complementary)
2,조솔림_1,0.8346,0.8108,0.6767,FALTP,SHETG,False,보완형(Complementary)
3,안은찬,0.8193,0.8108,0.6643,FALTP,SHEBG,True,보완형(Complementary)





🎯 강린지님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: SHETP
💕 당신의 이상형: 강아지상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


...

,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,한호주,0.9359,0.9191,0.8603,SHETP,FHLBP,False,보완형(Complementary)
2,남민나_1,0.9641,0.8143,0.7850,SHETP,FALBG,False,보완형(Complementary)
3,고연도,0.9641,0.7395,0.7129,SHETP,FHLBP,False,보완형(Complementary)





🎯 강태성님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FHLTG
💕 당신의 이상형: 곰상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,한준라,1.0,0.9978,0.9994,SHLTG,False,유사형(Similar)
2,전연라,1.0,0.9920,0.9976,FHLTG,False,유사형(Similar)
3,허태태,1.0,0.9667,0.9900,SHLTG,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,신하하_1,0.8165,0.8979,0.7331,FHLTG,FHEBP,True,보완형(Complementary)
2,전현별_1,1.0000,0.6915,0.6915,FHLTG,FHLTG,False,보완형(Complementary)
3,남겸겸,0.8347,0.8108,0.6768,FHLTG,SHLBP,False,보완형(Complementary)


## 상세 분석: 두 사용자 간 매칭 상세 정보

In [28]:
def analyze_match_detail(user1, user2):
    """
    두 사용자 간의 매칭 상세 분석
    """
    if user1 not in df_features.index or user2 not in df_features.index:
        return "사용자를 찾을 수 없습니다."
    
    user1_iloc = df_features.index.get_loc(user1)
    user2_iloc = df_features.index.get_loc(user2)
    
    print("="*60)
    print(f"📊 {user1} ↔ {user2} 매칭 상세 분석")
    print("="*60)
    
    # 1. 가치관 유사도
    val_score = float(val_sim_df.iloc[user1_iloc, user2_iloc])
    print(f"\n🎯 가치관 유사도: {val_score:.2%}")
    
    # 2. 성향 비교
    trait_cols = FEATURE_GROUPS['trait_cols']
    if trait_cols:
        print(f"\n📈 양육 성향 비교:")
        print("-"*50)
        print(f"{'축':<25} {user1:<10} {user2:<10} {'차이':>10}")
        print("-"*50)
        
        for col in trait_cols:
            col_idx = df_features.columns.get_loc(col)
            v1 = float(df_features.iloc[user1_iloc, col_idx])
            v2 = float(df_features.iloc[user2_iloc, col_idx])
            diff = abs(v1 - v2)
            print(f"{col:<25} {v1:.3f}      {v2:.3f}      {diff:.3f}")
    
    # 3. MBTI 비교
    if 'childcare_mbti' in df_mbti.columns:
        mbti1 = str(df_mbti.iloc[user1_iloc]['childcare_mbti'])
        mbti2 = str(df_mbti.iloc[user2_iloc]['childcare_mbti'])
        print(f"\n🧬 자녀양육 MBTI:")
        print(f"   {user1}: {mbti1}")
        print(f"   {user2}: {mbti2}")
        
        # 같은 문자 개수
        same_count = sum(1 for a, b in zip(mbti1, mbti2) if a == b)
        print(f"   일치하는 축: {same_count}/{min(len(mbti1), len(mbti2))}")
    
    # 4. 이상형 비교
    if 'ideal_type' in df.columns:
        ideal1 = str(df.iloc[user1_iloc]['ideal_type'])
        ideal2 = str(df.iloc[user2_iloc]['ideal_type'])
        print(f"\n💕 이상형:")
        print(f"   {user1}: {ideal1}")
        print(f"   {user2}: {ideal2}")
    
    # 5. 종합 평가
    comp_score, _ = calculate_complementary_score(user1_iloc, user2_iloc, df_features, trait_cols)
    print(f"\n🏆 종합 평가:")
    print(f"   유사도 점수: {val_score:.2%}")
    print(f"   보완도 점수: {comp_score:.2%}")
    
    if val_score >= 0.7 and comp_score >= 0.5:
        print(f"   → ⭐⭐⭐ 최고의 매칭! (가치관 일치 + 적절한 보완)")
    elif val_score >= 0.5:
        print(f"   → ⭐⭐ 좋은 매칭 (가치관 유사)")
    elif comp_score >= 0.5:
        print(f"   → ⭐⭐ 보완적 매칭 (성향 보완)")
    else:
        print(f"   → ⭐ 보통 매칭")
    
    print("="*60)

# 테스트
users = list(df_features.index[:2])
if len(users) >= 2:
    analyze_match_detail(users[0], users[1])

📊 윤서나 ↔ 권은준 매칭 상세 분석

🎯 가치관 유사도: 50.78%

📈 양육 성향 비교:
--------------------------------------------------
축                         윤서나        권은준                차이
--------------------------------------------------
parenting_style_SF        0.875      0.625      0.250
education_priority_AH     0.500      0.500      0.000
co_parenting_mode_EL      0.125      0.750      0.625
family_boundary_BT        0.750      0.750      0.000
planning_risk_PG          0.500      0.500      0.000

🧬 자녀양육 MBTI:
   윤서나: FAETP
   권은준: FALTP
   일치하는 축: 4/5

💕 이상형:
   윤서나: 오리상
   권은준: 고양이상

🏆 종합 평가:
   유사도 점수: 50.78%
   보완도 점수: 40.14%
   → ⭐⭐ 좋은 매칭 (가치관 유사)


# 7. 데이터 저장 및 요약

In [29]:
# 처리된 피처 데이터 저장
output_features_path = "processed_features.csv"
df_features.to_csv(output_features_path)
print(f"피처 데이터 저장: {output_features_path}")

# MBTI 데이터 저장
output_mbti_path = "childcare_mbti.csv"
df_mbti.to_csv(output_mbti_path)
print(f"MBTI 데이터 저장: {output_mbti_path}")

# 유사도 매트릭스 저장
output_sim_path = "similarity_matrix.csv"
val_sim_df.to_csv(output_sim_path)
print(f"유사도 매트릭스 저장: {output_sim_path}")

피처 데이터 저장: processed_features.csv
MBTI 데이터 저장: childcare_mbti.csv
유사도 매트릭스 저장: similarity_matrix.csv


In [30]:
print("\n" + "="*70)
print("📊 최종 요약")
print("="*70)
print(f"\n총 사용자 수: {len(df_features)}명")
print(f"피처 수: {df_features.shape[1]}개")
print(f"  - 가치관 피처: {len(FEATURE_GROUPS['value_cols'])}개")
print(f"  - 성향 피처: {len(FEATURE_GROUPS['trait_cols'])}개")

print(f"\n자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts().head(10))

print(f"\n이상형 분포:")
if 'ideal_type' in df.columns:
    print(df['ideal_type'].value_counts())

print("\n" + "="*70)
print("✅ 추천 시스템 구축 완료!")
print("="*70)


📊 최종 요약

총 사용자 수: 9966명
피처 수: 34개
  - 가치관 피처: 29개
  - 성향 피처: 5개

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    1006
SHLBP     773
SHEBG     749
SHEBP     712
SHLTG     554
FHLTG     501
FHEBP     478
SHETG     477
FHLBG     462
FHLBP     460
Name: count, dtype: int64

이상형 분포:
ideal_type
여우상      2383
곰상       1621
고양이상     1587
강아지상     1314
토끼상       550
오리상       512
꽃돼지상      256
꼬부기상      248
쿼카상       247
너구리상      237
호랑이상      110
햄스터상      107
다람쥐상      105
공룡상       105
판다상       102
사슴상        99
사자상        99
고슴도치상      98
늑대상        95
펭귄상        91
Name: count, dtype: int64

✅ 추천 시스템 구축 완료!
